# Columnas calculadas para el catálogo de colaboradores de OTC

**Alexis Arrazola Jimenez** · A01664015
Soporte analítico para la toma de decisiones · Reto OT Consulting (OTC)

**Objetivo:** a partir del catálogo de colaboradores del ERP de OTC, crear columnas que el sistema todavía no calcula (costo por hora, diario, anual, costo interno e indirecto, edad y antigüedad), marcar inconsistencias de nómina y cruzar el resultado con las asignaciones para obtener el **costo mensual por cliente y proyecto**.

Todo sale de los datos que OTC ya tiene en su ERP; no se captura nada nuevo.

## 1. Carga de datos

Se usan dos archivos del ERP: `Catalogo_Colaboradores.xlsx` y `Asignaciones_Colaboradores.xlsx`. Si no están en la carpeta (por ejemplo, en una sesión nueva de Colab), la función abre el diálogo para subirlos.

Para no exponer datos personales, aquí solo se muestran el tamaño y las columnas de la tabla, no sus filas.

In [1]:
import os
import pandas as pd

FECHA_CORTE = pd.Timestamp("2026-11-18")   # fecha fija para que edad y antigüedad sean reproducibles

def cargar(archivo):
    """Lee un Excel; en Colab, si no existe, pide subirlo."""
    if not os.path.exists(archivo):
        try:
            from google.colab import files
            print(f"Sube {archivo}")
            subidos = files.upload()
            archivo = next(iter(subidos))   # por si Colab lo renombra, p. ej. "(1)"
        except ImportError:
            raise FileNotFoundError(f"Pon {archivo} en la misma carpeta que este notebook")
    return pd.read_excel(archivo)

colaboradores = cargar("Catalogo_Colaboradores.xlsx")
asignaciones = cargar("Asignaciones_Colaboradores.xlsx")

df = colaboradores.copy()   # trabajamos sobre una copia para no tocar el original
print(f"Colaboradores: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Asignaciones: {asignaciones.shape[0]} filas, {asignaciones.shape[1]} columnas")
print(list(df.columns))

Colaboradores: 107 filas, 45 columnas
Asignaciones: 31 filas, 11 columnas
['ID', 'Nombre Completo', 'Género', 'Fecha Nacimiento', 'Teléfono', 'Correo Electrónico', 'Ciudad', 'Domicilio', 'RFC', 'CURP', 'NSS', 'Estatus', 'Fecha Incorporación', 'Fecha de Baja', 'Tipo Nómina', 'Departamento', 'Puesto', 'Sueldo Neto IMSS', 'Sueldo Asimilados/Sindicato', 'Sueldo Complementos', 'Sueldo Neto Total', 'Comisión Nómina', 'Impuestos IMSS', 'Sueldo Total Integrado', 'SGMM', 'Seguro Vida', 'Beneficiario', 'Parentesco', 'Porcentaje Beneficiario', 'Alergias/Condiciones', 'Notas', 'Doc: Acta', 'Doc: INE', 'Doc: CURP', 'Doc: Pasaporte', 'Doc: Visa', 'Doc: Const. Situación Fiscal', 'Doc: NSS', 'Doc: Comprobante Domicilio', 'Doc: Banco', 'Doc: Foto', 'Doc: Contrato', 'Aviso: Infonavit', 'Aviso: Manutención', 'Formato: RH']


In [2]:
print(df["Estatus"].value_counts())

Estatus
Activo    75
Baja      32
Name: count, dtype: int64


### Cómo se arma el sueldo

Antes de calcular, comprobamos cómo se relacionan las columnas de sueldo:

- **Sueldo Neto Total** = Sueldo Neto IMSS + Sueldo Asimilados/Sindicato + Sueldo Complementos
- **Sueldo Total Integrado** = Sueldo Neto Total + Comisión Nómina + Impuestos IMSS

Si ambas igualdades se cumplen, *Sueldo Total Integrado* es el costo mensual completo de cada colaborador para OTC.

In [3]:
SUELDOS = ["Sueldo Neto IMSS", "Sueldo Asimilados/Sindicato", "Sueldo Complementos",
           "Sueldo Neto Total", "Comisión Nómina", "Impuestos IMSS", "Sueldo Total Integrado"]
for col in SUELDOS:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

neto = df["Sueldo Neto IMSS"] + df["Sueldo Asimilados/Sindicato"] + df["Sueldo Complementos"]
integrado = df["Sueldo Neto Total"] + df["Comisión Nómina"] + df["Impuestos IMSS"]

print("Neto Total cuadra:", (neto - df["Sueldo Neto Total"]).abs().lt(1).sum(), "de", len(df))
print("Total Integrado cuadra:", (integrado - df["Sueldo Total Integrado"]).abs().lt(1).sum(), "de", len(df))
print("Colaboradores con pago por asimilados/sindicato:", df["Sueldo Asimilados/Sindicato"].gt(0).sum())

Neto Total cuadra: 107 de 107
Total Integrado cuadra: 107 de 107
Colaboradores con pago por asimilados/sindicato: 68


Las dos igualdades se cumplen para todos los registros. Un detalle importante: la mayoría de los colaboradores recibe parte de su pago como **asimilados/sindicato**. Si el costo interno se calcula solo con *IMSS + Complementos*, esa parte queda fuera y el costo sale muy por debajo del real. Por eso aquí `CostoInterno` incluye las tres partes del sueldo neto.

## 2. Columnas nuevas

**Supuestos** (no vienen en los datos, los definimos nosotros):
- Jornada de 8 horas × 20 días hábiles = **160 horas al mes**.
- Edad y antigüedad se calculan a la fecha de corte del análisis (18-nov-2026).

| Columna | Cálculo | Para qué sirve |
|---|---|---|
| `CostoHora` | Sueldo Total Integrado / 160 | Costear horas de proyecto |
| `CostoDiario` | Sueldo Total Integrado / 20 | Costear asignaciones por días |
| `CostoAnual` | Sueldo Total Integrado × 12 | Presupuesto anual de nómina |
| `CostoInterno` | IMSS + Asimilados/Sindicato + Complementos | Lo que recibe el colaborador |
| `CostoIndirecto` | Comisión Nómina + Impuestos IMSS | Lo que OTC paga por encima del sueldo neto |
| `PctIndirecto` | CostoIndirecto / Sueldo Total Integrado | Peso del costo indirecto según el esquema de nómina |
| `Edad`, `AntiguedadAnios` | A partir de Fecha Nacimiento y Fecha Incorporación | Perfil de la plantilla |
| `Alerta_*` | Reglas de calidad de datos | Detectar registros que hay que corregir |

In [4]:
HORAS_MES = 160
DIAS_MES = 20

def anios_hasta_corte(fechas):
    fechas = pd.to_datetime(fechas, errors="coerce")
    return ((FECHA_CORTE - fechas).dt.days / 365.25).round(1)

def agregar_columnas(datos):
    datos = datos.copy()
    integrado = datos["Sueldo Total Integrado"]

    # Costos
    datos["CostoHora"] = integrado / HORAS_MES
    datos["CostoDiario"] = integrado / DIAS_MES
    datos["CostoAnual"] = integrado * 12
    datos["CostoInterno"] = (datos["Sueldo Neto IMSS"] + datos["Sueldo Asimilados/Sindicato"]
                             + datos["Sueldo Complementos"])
    datos["CostoIndirecto"] = datos["Comisión Nómina"] + datos["Impuestos IMSS"]
    datos["PctIndirecto"] = (datos["CostoIndirecto"] / integrado.where(integrado > 0)).round(4)

    # Perfil
    datos["Edad"] = anios_hasta_corte(datos["Fecha Nacimiento"])
    datos["AntiguedadAnios"] = anios_hasta_corte(datos["Fecha Incorporación"])

    # Alertas de calidad de datos
    activo = datos["Estatus"].eq("Activo")
    baja = datos["Estatus"].eq("Baja")
    con_fecha_baja = datos["Fecha de Baja"].notna()
    datos["Alerta_ActivoSinSueldo"] = activo & ~integrado.gt(0)
    datos["Alerta_BajaConSueldo"] = baja & integrado.gt(0)
    datos["Alerta_BajaSinFecha"] = baja & ~con_fecha_baja
    datos["Alerta_ActivoConFechaBaja"] = activo & con_fecha_baja
    datos["Alerta_PuestoPendiente"] = activo & datos["Puesto"].astype(str).str.upper().eq("PENDIENTE")
    return datos

df = agregar_columnas(df)

## 3. Limpieza

Quitamos las columnas personales o sensibles que no aportan al análisis: contacto, identificaciones oficiales, beneficiarios, condiciones de salud y fecha de nacimiento (ya se convirtió en `Edad`). El **ID** se queda porque es la llave para cruzar con las demás tablas.

In [5]:
COLUMNAS_PERSONALES = ["Teléfono", "Correo Electrónico", "Domicilio", "RFC", "CURP", "NSS", "Notas",
                       "Fecha Nacimiento", "Beneficiario", "Parentesco", "Porcentaje Beneficiario",
                       "Alergias/Condiciones"]

def limpiar(datos):
    quitar = [c for c in COLUMNAS_PERSONALES if c in datos.columns]
    print(f"Columnas eliminadas ({len(quitar)}):", quitar)
    return datos.drop(columns=quitar)

df = limpiar(df)

NUEVAS = ["CostoHora", "CostoDiario", "CostoAnual", "CostoInterno", "CostoIndirecto",
          "PctIndirecto", "Edad", "AntiguedadAnios"]
df[["ID", "Estatus", "Tipo Nómina"] + NUEVAS].head().round(2)

Columnas eliminadas (12): ['Teléfono', 'Correo Electrónico', 'Domicilio', 'RFC', 'CURP', 'NSS', 'Notas', 'Fecha Nacimiento', 'Beneficiario', 'Parentesco', 'Porcentaje Beneficiario', 'Alergias/Condiciones']


,ID,Estatus,Tipo Nómina,CostoHora,CostoDiario,CostoAnual,CostoInterno,CostoIndirecto,PctIndirecto,Edad,AntiguedadAnios
0,OTC003,Activo,XERES OT-SINDICATO,373.32,2986.54,716768.40,56500.00,3230.70,0.05,41.3,10.4
1,OTC004,Activo,XERES OT-SINDICATO,371.23,2969.86,712767.36,56166.58,3230.70,0.05,44.0,4.3
2,OTC005,Activo,OTC-SINDICATO (SMR),419.18,3353.46,804830.16,60357.34,6711.84,0.10,51.9,10.9
3,OTC006,Activo,SMR,409.46,3275.67,786160.20,58660.87,6852.48,0.10,55.6,10.7
4,OTC007,Baja,XERES OT-SINDICATO,344.11,2752.92,660699.60,47500.00,7558.30,0.14,33.5,1.2


## 4. Estadísticas de referencia

Para los promedios usamos solo colaboradores **activos con sueldo registrado**: los sueldos en 0 bajarían el promedio sin representar un costo real.

In [6]:
activos = df[df["Estatus"].eq("Activo") & df["Sueldo Total Integrado"].gt(0)]

resumen = pd.Series({
    "Activos con sueldo": len(activos),
    "Costo hora promedio": activos["CostoHora"].mean(),
    "Costo diario promedio": activos["CostoDiario"].mean(),
    "Costo interno promedio": activos["CostoInterno"].mean(),
    "Costo indirecto promedio": activos["CostoIndirecto"].mean(),
    "Nómina mensual activa": activos["Sueldo Total Integrado"].sum(),
    "Nómina anual activa": activos["CostoAnual"].sum(),
    "Edad promedio": activos["Edad"].mean(),
    "Antigüedad promedio (años)": activos["AntiguedadAnios"].mean(),
}).round(2)
resumen.apply(lambda v: f"{v:,.2f}")

Activos con sueldo                    70.00
Costo hora promedio                  349.90
Costo diario promedio              2,799.19
Costo interno promedio            46,802.45
Costo indirecto promedio           9,181.32
Nómina mensual activa          3,918,864.40
Nómina anual activa           47,026,372.80
Edad promedio                         42.67
Antigüedad promedio (años)             5.67
dtype: str

In [7]:
# Costo por hora y peso del costo indirecto según el tipo de nómina
(activos.groupby("Tipo Nómina")
        .agg(Personas=("ID", "count"),
             CostoHoraPromedio=("CostoHora", "mean"),
             PctIndirectoPromedio=("PctIndirecto", "mean"))
        .sort_values("Personas", ascending=False)
        .round(2))

,Personas,CostoHoraPromedio,PctIndirectoPromedio
Tipo Nómina,,,
SMR,28,292.35,0.15
OTC 100%,13,397.96,0.34
OTC-SINDICATO (SMR),10,398.98,0.09
MAGNESON-SINDICATO,10,331.09,0.15
XERES OT-SINDICATO,6,456.92,0.06
MODALIDAD 40-SMR,3,363.79,0.05


In [8]:
alertas = [c for c in df.columns if c.startswith("Alerta")]
print("Registros con alerta:")
print(df[alertas].sum().to_string())

bajas = df[df["Estatus"].eq("Baja")]
print("\nBajas por mes (solo las que tienen fecha):")
print(pd.to_datetime(bajas["Fecha de Baja"], errors="coerce").dt.to_period("M").value_counts().sort_index().to_string())

Registros con alerta:
Alerta_ActivoSinSueldo        5
Alerta_BajaConSueldo         12
Alerta_BajaSinFecha          18
Alerta_ActivoConFechaBaja     1
Alerta_PuestoPendiente       29

Bajas por mes (solo las que tienen fecha):
Fecha de Baja
2026-08     1
2026-10    12
2026-11     1
Freq: M


## 5. Costo mensual por cliente y proyecto

Cruzamos el catálogo con la tabla de asignaciones por ID de colaborador. Si una persona aparece en más de un proyecto, su costo se **reparte en partes iguales** entre sus asignaciones, para no contarlo dos veces.

In [9]:
asg = asignaciones.merge(df[["ID", "Sueldo Total Integrado", "CostoHora"]],
                         left_on="ID Colaborador", right_on="ID", how="left")

asg["NumAsignaciones"] = asg.groupby("ID Colaborador")["ID Colaborador"].transform("size")
asg["CostoMensualAsignado"] = asg["Sueldo Total Integrado"] / asg["NumAsignaciones"]

print("Asignaciones sin colaborador en el catálogo:", asg["ID"].isna().sum())
print("Colaboradores en más de un proyecto:", asg.loc[asg["NumAsignaciones"] > 1, "ID Colaborador"].nunique())

costo_proyecto = (asg.groupby(["Cliente", "Proyecto"])
                     .agg(Personas=("ID Colaborador", "nunique"),
                          CostoMensual=("CostoMensualAsignado", "sum"),
                          CostoHoraPromedio=("CostoHora", "mean"))
                     .sort_values("CostoMensual", ascending=False)
                     .round(2))
costo_proyecto

Asignaciones sin colaborador en el catálogo: 0
Colaboradores en más de un proyecto: 1


Personas  CostoMensual  CostoHoraPromedio
Cliente        Proyecto                                                        
METALSA        Asignación indefinida        18    1268863.73             458.10
               Soporte AMS Manufacta         5     318535.35             398.17
Grupo Daltile  Marazzi                       3     172382.28             464.27
Allegion       Migración eBs                 5     164929.26             206.16

In [10]:
activos_ids = set(df.loc[df["Estatus"].eq("Activo"), "ID"])
con_asignacion = activos_ids & set(asg["ID Colaborador"])
print(f"Activos con asignación registrada: {len(con_asignacion)} de {len(activos_ids)} "
      f"({len(con_asignacion) / len(activos_ids):.0%})")

Activos con asignación registrada: 30 de 75 (40%)


## 6. Guardar resultados

Se generan dos archivos para seguir trabajando con ellos. **No se suben al repositorio**, porque contienen información del socio formador.

In [11]:
df.to_excel("Colaboradores_actualizado.xlsx", index=False)
costo_proyecto.to_excel("Costo_por_proyecto.xlsx")
print("Archivos guardados: Colaboradores_actualizado.xlsx, Costo_por_proyecto.xlsx")

try:
    from google.colab import files
    files.download("Colaboradores_actualizado.xlsx")
    files.download("Costo_por_proyecto.xlsx")
except ImportError:
    pass

Archivos guardados: Colaboradores_actualizado.xlsx, Costo_por_proyecto.xlsx


## 7. Conclusiones

- **Costo de la plantilla.** Los 70 activos con sueldo cuestan a OTC unos **$3.92 millones al mes**, con un costo promedio de **~$350 por hora**. Con `CostoHora` y `CostoDiario`, OTC ya puede costear cualquier asignación sin capturar nada nuevo.
- **El costo interno debe incluir asimilados/sindicato.** Sumar solo IMSS + complementos deja fuera la parte principal del pago de la mayoría de los colaboradores; el sueldo neto completo cuadra exactamente con *Sueldo Neto Total*.
- **Costo por proyecto.** La plantilla en Metalsa "Asignación indefinida" cuesta cerca de **$1.27 millones al mes**, la mayor concentración de costo. Al ser un proyecto comodín, ese costo no se puede ligar a un entregable concreto.
- **Cobertura baja.** Solo **30 de 75 activos (40 %)** tienen una asignación registrada, así que el costo por proyecto todavía no representa toda la nómina.
- **Nómina y estatus no siempre coinciden.** Hay **5 activos con sueldo 0**, **12 bajas que conservan sueldo** y **18 bajas sin fecha**. De las bajas con fecha, la mayoría se concentra en octubre de 2026. Conviene corregir estos registros antes de usar los costos para decidir.
- **El esquema de nómina cambia el costo.** En los activos con nómina *OTC 100%* el costo indirecto (comisión + impuestos) pesa alrededor del 34 % del costo integrado, contra 5–15 % en los esquemas sindicalizados. Las categorías de nómina además se solapan (SMR, OTC-SINDICATO (SMR), MODALIDAD 40-SMR), por lo que conviene estandarizarlas.
- **Supuesto de 160 horas.** Es una simplificación; si OTC trabaja con horas facturables o con otra jornada, basta con cambiar `HORAS_MES`.
- **Atributos que faltan en el ERP:** centro de costos, tarifa de venta por hora y fecha de baja siempre capturada. Con la tarifa de venta se podría calcular el margen por proyecto.